In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd(), '| alphas of record:', [config.ALPHA_PRIMARY]+config.ALPHA_SENSITIVITY)


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research | alphas of record: [0.05, 0.1, 0.2]


In [2]:
# =============================================================================
# Cell 2 - assemble focal coverage at every alpha for all three datasets.
# The claim: the focal SHC failure is not an artifact of alpha=0.05; it holds at
# 0.10 and 0.20 too. Focal per dataset: NSL R2L (rung 0.80, matching nb22),
# CIC DoS, UGR nerisbotnet. All from committed tables / the NSL long parquet.
# CIC/UGR coverage CSVs already carry alpha in {0.05,0.10,0.20}.
# =============================================================================
ALPHAS=[config.ALPHA_PRIMARY]+config.ALPHA_SENSITIVITY   # 0.05, 0.10, 0.20
rows=[]

def add(dataset, focal, sub):
    # sub: rows for the focal class at one dataset, columns include alpha, protocol, coverage
    for a in ALPHAS:
        g=sub[np.isclose(sub['alpha'],a)]
        if len(g)==0: continue
        piv=g.groupby('protocol')['coverage'].mean()
        shc=float(piv.get('SHC',np.nan)); tsc=float(piv.get('TSC',np.nan)); rec=float(piv.get('REC',np.nan))
        rows.append({'dataset':dataset,'focal_class':focal,'alpha':a,'nominal':round(1-a,3),
                     'REC':round(rec,4),'TSC':round(tsc,4),'SHC':round(shc,4),
                     'focal_gap_TSC_minus_SHC':round(tsc-shc,4),
                     'SHC_undercovers':bool(shc < (1-a)-0.02)})

# ---- NSL: long parquet has all alphas; focal R2L at rung 0.80 ----
nsl_long=config.PROC_DIR/'coverage_long_nslkdd.parquet'
if nsl_long.exists():
    nl=pd.read_parquet(nsl_long)
    nf=nl[(nl['class']=='R2L')&(nl['score']=='aps')&(nl['variant']=='mondrian')&
          (nl['feasible'])&(np.isclose(nl['rung'],0.80))].copy()
    add('nslkdd (R2L, rung0.80)','R2L',nf)
    print('NSL loaded from long parquet | alphas present:', sorted(nf['alpha'].unique()))
else:
    print('WARNING: coverage_long_nslkdd.parquet not found on Drive; NSL alpha-sensitivity skipped.')

# ---- CIC: committed CSV, focal DoS ----
cc=pd.read_csv(config.REPORTS_DIR/'coverage_primary_cicids2017.csv')
add('cicids2017 (DoS)','DoS',cc[cc['class']=='DoS'].copy())
print('CIC alphas present:', sorted(cc['alpha'].unique()))

# ---- UGR: committed CSV, focal nerisbotnet ----
uu=pd.read_csv(config.REPORTS_DIR/'coverage_primary_ugr16.csv')
add('ugr16 (nerisbotnet)','nerisbotnet',uu[uu['class']=='nerisbotnet'].copy())
print('UGR alphas present:', sorted(uu['alpha'].unique()))

alpha_tbl=pd.DataFrame(rows)
print('\nFOCAL coverage across alpha:')
print(alpha_tbl.to_string(index=False))


NSL loaded from long parquet | alphas present: [np.float64(0.01), np.float64(0.05), np.float64(0.1), np.float64(0.2)]
CIC alphas present: [np.float64(0.05), np.float64(0.1), np.float64(0.2)]
UGR alphas present: [np.float64(0.05), np.float64(0.1), np.float64(0.2)]

FOCAL coverage across alpha:
               dataset focal_class  alpha  nominal    REC    TSC    SHC  focal_gap_TSC_minus_SHC  SHC_undercovers
nslkdd (R2L, rung0.80)         R2L   0.05     0.95 0.9563 0.9531 0.0298                   0.9233             True
nslkdd (R2L, rung0.80)         R2L   0.10     0.90 0.9057 0.9026 0.0231                   0.8795             True
nslkdd (R2L, rung0.80)         R2L   0.20     0.80 0.8046 0.8036 0.0168                   0.7868             True
      cicids2017 (DoS)         DoS   0.05     0.95 0.9875 0.9876 0.6039                   0.3837             True
      cicids2017 (DoS)         DoS   0.10     0.90 0.9731 0.9732 0.5565                   0.4167             True
      cicids2017 (DoS)

In [3]:
# =============================================================================
# Cell 3 - UGR discovered finding across alpha (the scans), then verdict + save.
# UGR's preregistered focal (nerisbotnet) HOLDS; the collapse is in scan11/scan44,
# so alpha-sensitivity for UGR must also show the scans collapse at every alpha.
# =============================================================================
scan_rows=[]
for cls in ['scan11','scan44']:
    sub=uu[uu['class']==cls]
    for a in ALPHAS:
        g=sub[np.isclose(sub['alpha'],a)]
        if len(g)==0: continue
        piv=g.groupby('protocol')['coverage'].mean()
        scan_rows.append({'class':cls,'alpha':a,'nominal':round(1-a,3),
                          'SHC':round(float(piv.get('SHC',np.nan)),4),
                          'TSC':round(float(piv.get('TSC',np.nan)),4),
                          'SHC_undercovers':bool(float(piv.get('SHC',1))< (1-a)-0.02)})
scan_tbl=pd.DataFrame(scan_rows)
print('UGR scan classes across alpha (the discovered collapse):')
print(scan_tbl.to_string(index=False))

# verdict: does each focal SHC failure hold at ALL alphas?
holds={}
for ds,g in alpha_tbl.groupby('dataset'):
    # "failure holds" if SHC undercovers at every alpha (for collapsing focals),
    # OR SHC holds at every alpha (for UGR's nerisbotnet, which is meant to hold)
    under=g['SHC_undercovers'].tolist()
    holds[ds]={'alphas':g['alpha'].tolist(),'SHC':g['SHC'].tolist(),'nominal':g['nominal'].tolist(),
               'undercovers_each_alpha':under,'consistent':bool(all(under) or not any(under))}
scan_hold=bool(scan_tbl.groupby('class')['SHC_undercovers'].all().all()) if len(scan_tbl) else None

verdict={'analysis':'alpha-sensitivity: focal coverage at alpha in {0.05,0.10,0.20}',
   'focal_by_dataset':holds,
   'ugr_scans_collapse_every_alpha':scan_hold,
   'reads':('for NSL R2L and CIC DoS, SHC undercovers at every alpha => the failure is not an artifact of '
            'alpha=0.05. For UGR the preregistered focal nerisbotnet holds at every alpha, while scan11/scan44 '
            'collapse at every alpha => the selective covariate finding is also alpha-robust.')}
alpha_tbl.to_csv(config.REPORTS_DIR/'alpha_sensitivity.csv',index=False)
scan_tbl.to_csv(config.REPORTS_DIR/'alpha_sensitivity_ugr_scans.csv',index=False)
(config.REPORTS_DIR/'alpha_sensitivity_verdict.json').write_text(json.dumps(verdict,indent=2))
print('\n',json.dumps(verdict,indent=2))


UGR scan classes across alpha (the discovered collapse):
 class  alpha  nominal    SHC    TSC  SHC_undercovers
scan11   0.05     0.95 0.5350 0.9502             True
scan11   0.10     0.90 0.4879 0.9005             True
scan11   0.20     0.80 0.4126 0.8004             True
scan44   0.05     0.95 0.7990 0.9820             True
scan44   0.10     0.90 0.7506 0.9576             True
scan44   0.20     0.80 0.6646 0.8005             True

 {
  "analysis": "alpha-sensitivity: focal coverage at alpha in {0.05,0.10,0.20}",
  "focal_by_dataset": {
    "cicids2017 (DoS)": {
      "alphas": [
        0.05,
        0.1,
        0.2
      ],
      "SHC": [
        0.6039,
        0.5565,
        0.4786
      ],
      "nominal": [
        0.95,
        0.9,
        0.8
      ],
      "undercovers_each_alpha": [
        true,
        true,
        true
      ],
      "consistent": true
    },
    "nslkdd (R2L, rung0.80)": {
      "alphas": [
        0.05,
        0.1,
        0.2
      ],
      "SHC": 

In [ ]:
# =============================================================================
# Cell 4 - figure + commit.
# =============================================================================
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(6.4,4.6))
mark={'nslkdd (R2L, rung0.80)':'^','cicids2017 (DoS)':'o','ugr16 (nerisbotnet)':'s'}
for ds,g in alpha_tbl.groupby('dataset'):
    g=g.sort_values('alpha')
    ax.plot(g['alpha'],g['SHC'],marker=mark.get(ds,'x'),label=f'{ds} SHC')
# nominal line 1-alpha
xa=np.array(sorted(ALPHAS)); ax.plot(xa,1-xa,'k--',lw=0.8,label='nominal (1-alpha)')
ax.set_xlabel('alpha'); ax.set_ylabel('focal SHC coverage'); ax.set_xticks(sorted(ALPHAS))
ax.set_title('Focal SHC coverage vs nominal, across alpha'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(config.REPORTS_DIR/'alpha_sensitivity.png',dpi=140); print('figure saved')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb24: alpha-sensitivity - focal failures hold across alpha {0.05,0.10,0.20} on all 3 datasets')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


figure saved
